# Market strategy hyperparameter sweep — Colab\nJoint grid over **Lasso α × learning-rate η × vol-target**, per market, walk-forward out-of-sample.\n\n**Runtime → Run all.** Pulls code + 836 MB of 10y panels from a public GitHub repo (Git LFS, no login), runs the sweep, prints the ROE-optimal (α, η, vol-target) per desk. ~15–25 min on free Colab CPU.\n\n_Data source: github.com/herrrickshaw/global-market-data @ colab-data. Descriptive backtest research — not investment advice._

In [ ]:
# --- pull code + market data from GitHub (public repo: herrrickshaw/global-market-data @ colab-data) ---
import os, subprocess, urllib.request, time
CODE_URL = 'https://raw.githubusercontent.com/herrrickshaw/global-market-data/colab-data/code.tgz'       # code bundle (regular git)
DATA_URL = 'https://media.githubusercontent.com/media/herrrickshaw/global-market-data/colab-data/warehouse.tar.gz'       # 836 MB warehouse (Git LFS media URL)
t0=time.time()
print("downloading code ..."); urllib.request.urlretrieve(CODE_URL, "code.tgz")
print("downloading market data (~836 MB via LFS) ..."); urllib.request.urlretrieve(DATA_URL, "warehouse.tar.gz")
subprocess.run(["tar","xzf","code.tgz"], check=True)
subprocess.run("mkdir -p data && tar xzf warehouse.tar.gz -C data", shell=True, check=True)
print(f"ready in {time.time()-t0:.0f}s; markets:", sorted(os.listdir("data/ohlcv")))
!pip -q install pyarrow scikit-learn 2>/dev/null | tail -1

In [ ]:
# --- run the 180-cell hyperparameter sweep ---
import os
os.environ["MARKET_WH"]=os.path.abspath("data/ohlcv")
os.environ["MARKET_LOGDIR"]=os.path.abspath("logs")
os.environ["SWEEP_OUT"]=os.path.abspath("aws_sweep.parquet")
!MARKET_WH="$MARKET_WH" MARKET_LOGDIR="$MARKET_LOGDIR" SWEEP_OUT="$SWEEP_OUT" python aws_sweep.py

In [ ]:
# --- best hyperparameters per market ---
import pandas as pd
df=pd.read_parquet("aws_sweep.parquet")
best=df.loc[df.groupby("market")["oos_ir"].idxmax()].sort_values("oos_ir",ascending=False)
display(best[["market","alpha","eta","vol_target","oos_ir","oos_maxdd","oos_ann_ret"]].reset_index(drop=True))
best.to_csv("best_hyperparams.csv",index=False)